[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/semantica-agi/semantica/blob/main/cookbook/introduction/23_Reasoning.ipynb)

> **中文译文** —— 由社区翻译，非上游原文；源文件变更后以英文原版为准。步骤间的逻辑导读为译者补充。

# 推理模块（Reasoning）—— 实战指南

Semantica 的 `reasoning` 模块从既有事实和知识图谱推导新知识。一个门面背后藏着多种策略：

- **`Reasoner`** - 统一门面：前向链、反向链、一次性 `infer_facts`
- **`DatalogReasoner`** - 半朴素 Datalog 定点求值，支持变量查询
- **`ExplanationGenerator`** - 为推断结论生成人类可读的解释和推理路径
- 另有低层引擎：`ReteEngine`、`SPARQLReasoner`、`GraphReasoner`、时序推理

本 notebook 走读门面、Datalog 引擎和解释生成。所有 API 均对照 `semantica/reasoning/` 验证过。

### 先看主线

图谱存的是"知道什么"，推理回答"还能推出什么"。打个比方：前向链是老师把所有推论全写上黑板；反向链是学生问"这个结论对吗"，老师倒着找理由。六节三段——**推得出 → 证得了 → 讲得清**：

**推得出（第 1–2 节）—— 前向链**

1. 第 1 节 前向链：事实 + 规则滚雪球，推出一切可推出的
2. 第 2 节 一次性推理：`infer_facts` 传入即推（注意：它往实例里*追加*，不重置状态）

**证得了（第 3–4 节）—— 反向链**

3. 第 3 节 反向链：从结论倒推，返回证明或"不可证"
4. 第 4 节 重跑安全：重复添加同一规则自动去重——Jupyter 反复执行 cell 不出事

**讲得清（第 5–6 节）—— Datalog 与解释**

5. 第 5 节 Datalog：经典逻辑编程写法（`head :- body.`），定点求值 + 变量查询
6. 第 6 节 解释：`ExplanationGenerator` 把推断变成推理路径——22 篇给了事实以来源，这里给推断以理由，可信链条延伸到推理层

In [1]:
!pip install -q semantica

## 1) 用 `Reasoner` 门面做前向链

事实就是简单的 `Predicate(args)` 字符串。规则用 `IF <条件> THEN <结论>`，变量写成 `?x`。`forward_chain()` 推出一切可推出的，返回 `InferenceResult` 列表。

In [2]:
from semantica.reasoning import Reasoner

reasoner = Reasoner()

reasoner.add_fact("Person(John)")
reasoner.add_fact("Person(Jane)")
reasoner.add_rule("IF Person(?x) THEN Human(?x)")

results = reasoner.forward_chain()
print(f"Inferred {len(results)} new facts")
for res in results:
    print(f"  {res.conclusion}  (rule: {res.rule_used.name}, confidence: {res.confidence})")

Status,Action,Module,Submodule,Progress,ETA,Rate,Time,Extracted
✅,Semantica is reasoning,🤔 reasoning,Reasoner,100.0%,-,-,0.00s,-
✅,Semantica is reasoning,🤔 reasoning,DatalogReasoner,100.0%,-,-,0.00s,-
✅,Semantica is reasoning,🤔 reasoning,ExplanationGenerator,100.0%,-,-,0.00s,-


🔄 Semantica is reasoning: Performing forward chaining 🤔 reasoning Reasoner |░░░░░░░░░░░░░░░| 0.0% ETA: - Rate: - Time: 0.00s Extracted: -

Inferred 2 new facts
  Human(Jane)  (rule: Rule 1, confidence: 1.0)
  Human(John)  (rule: Rule 1, confidence: 1.0)


## 2) 用 `infer_facts` 做一次性推理

`infer_facts(facts, rules)` 把给定的事实和规则**追加**到这个 `Reasoner` 实例，跑到定点，把推导出的事实作为字符串返回。它不会重置实例的既有状态——需要隔离时先建一个全新的 `Reasoner()`。

In [3]:
from semantica.reasoning import Reasoner

derived = Reasoner().infer_facts(
    facts=["WorksFor(John, Acme)", "WorksFor(Jane, Acme)"],
    rules=["IF WorksFor(?x, ?y) THEN Employee(?x, ?y)"],
)
derived

['Employee(Jane, Acme)', 'Employee(John, Acme)']

## 3) 反向链：证明一个目标

`backward_chain(goal)` 从结论出发、沿规则倒着找。返回证明目标的 `InferenceResult`，证不出则返回 `None`。

In [4]:
from semantica.reasoning import Reasoner

reasoner = Reasoner()
reasoner.add_fact("Person(John)")
reasoner.add_rule("IF Person(?x) THEN Human(?x)")

proof = reasoner.backward_chain("Human(John)")
print(proof.conclusion if proof else "not provable")
print("premises:", proof.premises if proof else None)

Human(John)
premises: ['Person(John)']


## 4) 重跑安全

`add_rule` 对条件与结论完全相同的规则去重——重跑一个设置 cell（Jupyter 的常见操作）不会堆出重复规则。见 issue #732。

In [5]:
from semantica.reasoning import Reasoner

reasoner = Reasoner()
reasoner.add_fact("Person(John)")

# Simulate a Jupyter cell re-run: add the same rule twice
r1 = reasoner.add_rule("IF Person(?x) THEN Human(?x)")
r2 = reasoner.add_rule("IF Person(?x) THEN Human(?x)")

len(reasoner.rules)

Skipping duplicate rule (same conditions/conclusion as 'rule_1'): IF Person(?x) THEN Human(?x)


1

## 5) Datalog 推理

`DatalogReasoner` 用经典 Datalog 语法（`head :- body.`）和半朴素定点求值。查询返回变量绑定的字典列表——变量用大写，问的是"*哪些*事实成立"。

In [6]:
from semantica.reasoning import DatalogReasoner

datalog = DatalogReasoner()
datalog.add_fact("parent(tom, mary)")
datalog.add_fact("parent(mary, ann)")
datalog.add_rule("grandparent(X, Z) :- parent(X, Y), parent(Y, Z)")

datalog.derive_all()
datalog.query("grandparent(X, Z)")

[{'X': 'tom', 'Z': 'ann'}]

## 6) 推断结论的解释

`ExplanationGenerator` 把 `InferenceResult` 变成结构化的 `Explanation` 和 `ReasoningPath` 记录——智能体因此能展示*为什么*相信某条推导出的事实。

In [7]:
from semantica.reasoning import Reasoner, ExplanationGenerator

reasoner = Reasoner()
reasoner.add_fact("Person(John)")
reasoner.add_rule("IF Person(?x) THEN Human(?x)")
results = reasoner.forward_chain()

gen = ExplanationGenerator()
explanation = gen.generate_explanation(results[0])
path = gen.show_reasoning_path(results[0])

type(explanation).__name__, type(path).__name__

('Explanation', 'ReasoningPath')

## 小结

### 回顾主线

三段走完：**推得出**（前向链滚雪球、一次性推理）、**证得了**（反向链倒推证明、重跑安全）、**讲得清**（Datalog 定点求值、解释生成）。事实有来源（22 篇）、推断有路径（本篇）——图谱的可信因此完整。

| 任务 | API |
|---|---|
| 推导全部新事实 | `Reasoner.forward_chain()` |
| 一次性推理 | `Reasoner.infer_facts(facts, rules)` |
| 证明一个目标 | `Reasoner.backward_chain(goal)` |
| Datalog 定点 | `DatalogReasoner.derive_all()` + `query("p(X, Y)")` |
| 解释一个结论 | `ExplanationGenerator.generate_explanation(result)` |

另见 `semantica/reasoning/reasoning_usage.md` 及 `ReteEngine`、`SPARQLReasoner`、时序推理的模块 docstring。